# 01 — Pipeline Data Chat Dota 2 (2016-2026)

Pipeline ingest + clean + language filter + join metadata untuk seluruh folder `dota2_dataset_bersih/`. Output: `data/processed/<folder>.parquet`.

Tahap per-folder:
1. Load `chat.csv` + `main_metadata.csv`.
2. Drop chatwheel + normalisasi teks.
3. Filter bahasa Inggris (fasttext + short-circuit ≤ 5 token).
4. Join metadata + kolom turunan kontekstual.
5. Tulis parquet dengan idempotent hash check.

Re-run aman: folder yang sumber-nya tidak berubah otomatis di-skip via SHA256 sidecar.

In [1]:
%pip install pyarrow scipy statsmodels seaborn matplotlib scikit-learn pyyaml tqdm
!pip install langdetect

   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   -------- ------------------------------- 5.8/27.3 MB 32.0 MB/s eta 0:00:01
   -------------------- ------------------- 14.2/27.3 MB 37.0 MB/s eta 0:00:01
   --------------------------------- ------ 22.8/27.3 MB 38.0 MB/s eta 0:00:01
   ---------------------------------------- 27.3/27.3 MB 35.2 MB/s  0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ----------------------------------- ---- 8.4/9.6 MB 43.2 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 37.3 MB/s  0:00:00

   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [pyarrow]
   ---------------------------------------- 0/4 [


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sel 1: Setup — import, banner versi/seed, load config, init run log
import sys
from pathlib import Path

# Pastikan root repo ada di sys.path supaya `from src...` import berfungsi
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog

config = load_config('configs/experiment.yaml')
print_banner('01_data_pipeline', config)
run_log = RunLog(notebook='01_data_pipeline', config_path='configs/experiment.yaml')

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 01_data_pipeline
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-04-30T04:57:17+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: not-installed
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [3]:
# Sel 2: Konfigurasi pipeline
from pathlib import Path

RAW_ROOT = Path(config['data']['raw_root'])
PROCESSED_ROOT = Path(config['data']['processed_root'])
FOLDERS = list(config['folders'])
DETECTOR = config['language_filter']['detector']
SHORT_CIRCUIT_MAX = int(config['language_filter']['short_circuit_max_tokens'])

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Raw root        : {RAW_ROOT.resolve()}')
print(f'Processed root  : {PROCESSED_ROOT.resolve()}')
print(f'Folders ({len(FOLDERS)}): {FOLDERS}')
print(f'Detector        : {DETECTOR}')
print(f'Short-circuit   : ≤ {SHORT_CIRCUIT_MAX} token alfanumerik dianggap EN')

Raw root        : C:\#fileUtama\pre-thesis\dota2_dataset_bersih
Processed root  : C:\#fileUtama\pre-thesis\data\processed
Folders (14): ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '202601', '202602', '202603', '202604']
Detector        : langdetect
Short-circuit   : ≤ 5 token alfanumerik dianggap EN


In [4]:
# Sel 3: Loop per-folder — load → clean → lang_filter → join → enrich → write
from src.pipeline.loader import folder_path, folder_exists, load_chat, load_metadata
from src.pipeline.cleaner import clean_chat
from src.pipeline.lang_filter import filter_english
from src.pipeline.joiner import join_with_metadata
from src.pipeline.enrich import (
    build_league_tier_map, build_patch_map, enrich_dataframe,
)
from src.pipeline.writer import write_processed, write_non_english, sha256_of_sources
import pandas as pd

# Bangun maps sekali di awal (sumber: dota2_dataset_bersih/Constants/).
league_map_path = Path('data/processed/league_tier_map.csv')
patch_map_path = Path('data/processed/patch_map.csv')
if not league_map_path.exists() or not patch_map_path.exists():
    build_league_tier_map(RAW_ROOT / 'Constants', league_map_path)
    build_patch_map(RAW_ROOT / 'Constants', patch_map_path)
league_map = pd.read_csv(league_map_path)
patch_map = pd.read_csv(patch_map_path)
patch_map['date_start'] = pd.to_datetime(patch_map['date_start'])
patch_map['date_end'] = pd.to_datetime(patch_map['date_end'])
print(f'league_map: {len(league_map):,} rows  |  patch_map: {len(patch_map)} rows')

summary_rows = []

for folder in FOLDERS:
    if not folder_exists(RAW_ROOT, folder):
        msg = f'[SKIP] folder={folder} (tidak ditemukan di {RAW_ROOT})'
        print(msg)
        run_log.add_warning(msg)
        continue

    print(f'\n=== folder={folder} ===')
    chat_path = folder_path(RAW_ROOT, folder) / 'chat.csv'
    md_path = folder_path(RAW_ROOT, folder) / 'main_metadata.csv'
    src_hash = sha256_of_sources(chat_path, md_path)

    out_path = PROCESSED_ROOT / f'{folder}.parquet'
    out_non_en = PROCESSED_ROOT / f'{folder}_non_english.parquet'

    # Cek idempotent sebelum baca CSV besar (cepat).
    sidecar = out_path.with_suffix(out_path.suffix + '.meta.json')
    if out_path.exists() and sidecar.exists():
        import json
        try:
            meta = json.loads(sidecar.read_text(encoding='utf-8'))
            if meta.get('sources_hash') == src_hash:
                print(f'  [SKIP] cached (n_rows={meta.get("n_rows")})')
                summary_rows.append({'folder': folder, 'status': 'skipped', 'n_out': meta.get('n_rows', 0)})
                run_log.add_output(out_path)
                continue
        except Exception:
            pass

    chat = load_chat(RAW_ROOT, folder)
    md = load_metadata(RAW_ROOT, folder)
    print(f'  loaded   : chat={len(chat):,} rows, metadata={len(md):,} rows')

    chat_clean, clean_stats = clean_chat(chat)
    print(f'  cleaned  : chatwheel_dropped={clean_stats.n_chatwheel_dropped:,}  empty={clean_stats.n_empty_dropped:,}  → {clean_stats.n_out:,}')

    df_en, df_non_en, lang_stats = filter_english(
        chat_clean, text_col='key', detector=DETECTOR, short_circuit_max_tokens=SHORT_CIRCUIT_MAX
    )
    print(f'  lang     : short_circuit_en={lang_stats.n_short_circuit_en:,}  detected_en={lang_stats.n_detected_en:,}  non_en={lang_stats.n_non_en:,}')

    df_en, join_stats = join_with_metadata(df_en, md)
    df_non_en, _ = join_with_metadata(df_non_en, md)
    print(f'  joined   : with_metadata={join_stats.n_with_metadata:,}  missing={join_stats.n_missing_metadata:,}')

    df_en, en_stats = enrich_dataframe(df_en, league_map, patch_map)
    df_non_en, _ = enrich_dataframe(df_non_en, league_map, patch_map)
    print(f'  enriched : ti={en_stats.n_ti:,}  major={en_stats.n_major:,}  dpc={en_stats.n_dpc_tour:,}  lainnya={en_stats.n_lainnya:,}  patch_resolved={en_stats.n_patch_resolved:,}')

    w_en = write_processed(df_en, out_path, sources_hash=src_hash)
    w_non = write_non_english(df_non_en, out_non_en)
    print(f'  wrote    : {w_en.out_path} ({w_en.n_rows:,} rows, sha256={w_en.output_sha256[:12]}...)')
    print(f'  non_en   : {w_non.out_path} ({w_non.n_rows:,} rows)')
    run_log.add_output(out_path)
    run_log.add_output(out_non_en)

    summary_rows.append({
        'folder': folder,
        'status': 'ok',
        'n_in': clean_stats.n_in,
        'n_chatwheel': clean_stats.n_chatwheel_dropped,
        'n_empty': clean_stats.n_empty_dropped,
        'n_short_en': lang_stats.n_short_circuit_en,
        'n_det_en': lang_stats.n_detected_en,
        'n_non_en': lang_stats.n_non_en,
        'n_ti': en_stats.n_ti,
        'n_major': en_stats.n_major,
        'n_dpc': en_stats.n_dpc_tour,
        'n_out': w_en.n_rows,
        'output_sha256': w_en.output_sha256[:16],
    })

summary = pd.DataFrame(summary_rows)
summary

league_map: 8,812 rows  |  patch_map: 57 rows

=== folder=2016 ===
  loaded   : chat=144,336 rows, metadata=7,779 rows
  cleaned  : chatwheel_dropped=12,406  empty=44  → 131,886
  lang     : short_circuit_en=117,441  detected_en=3,013  non_en=11,432
  joined   : with_metadata=120,454  missing=0
  enriched : ti=3,199  major=4,319  dpc=0  lainnya=112,936  patch_resolved=120,454
  wrote    : data\processed\2016.parquet (120,454 rows, sha256=fc37c0880c60...)
  non_en   : data\processed\2016_non_english.parquet (11,432 rows)

=== folder=2017 ===
  loaded   : chat=207,571 rows, metadata=8,892 rows
  cleaned  : chatwheel_dropped=94,770  empty=48  → 112,753
  lang     : short_circuit_en=99,131  detected_en=2,681  non_en=10,941
  joined   : with_metadata=101,812  missing=0
  enriched : ti=4,318  major=3,694  dpc=0  lainnya=93,800  patch_resolved=101,812
  wrote    : data\processed\2017.parquet (101,812 rows, sha256=9653ea2290a6...)
  non_en   : data\processed\2017_non_english.parquet (10,941 ro

,folder,status,n_in,n_chatwheel,n_empty,n_short_en,n_det_en,n_non_en,n_ti,n_major,n_dpc,n_out,output_sha256
0,2016,ok,144336,12406,44,117441,3013,11432,3199,4319,0,120454,fc37c0880c60644c
1,2017,ok,207571,94770,48,99131,2681,10941,4318,3694,0,101812,9653ea2290a6bcde
2,2018,ok,364438,236322,55,112675,2191,13195,3839,8505,0,114866,4c6e167d4e311df1
3,2019,ok,763692,537937,58,184339,2733,38625,4014,9754,0,187072,4f1ed992a0d989ff
4,2020,ok,882130,691190,49,160584,2993,27314,0,0,0,163577,37d1f35c4caa6e55
5,2021,ok,878926,705043,25,149989,2369,21500,4846,2275,20947,152358,5c68ba2983ca4a3d
6,2022,ok,1073295,897674,39,154726,2499,18357,2270,3025,20468,157225,43f2eededffd9825
7,2023,ok,1515887,1308393,44,168338,2205,36907,1511,5394,21669,170543,93fc6ce4b95e6127
8,2024,ok,1208626,992534,96,179732,2842,33422,1275,845,0,182574,67a787b6d1824901
9,2025,ok,1236697,992368,271,195691,2090,46277,1520,0,0,197781,2b4a83f3f4805860


In [5]:
# Sel 4: Verifikasi — concat seluruh output processed harus mulus (schema stable)
import pandas as pd
from src.pipeline.writer import PROCESSED_SCHEMA

files = sorted(PROCESSED_ROOT.glob('*.parquet'))
files = [f for f in files if not f.stem.endswith('_non_english')]
print(f'Memverifikasi concat {len(files)} file processed...')
frames = [pd.read_parquet(f) for f in files]
all_df = pd.concat(frames, ignore_index=True)

missing = [c for c in PROCESSED_SCHEMA if c not in all_df.columns]
extra = [c for c in all_df.columns if c not in PROCESSED_SCHEMA]
assert not missing, f'Kolom hilang: {missing}'
assert not extra, f'Kolom ekstra: {extra}'
print(f'OK — total {len(all_df):,} baris di {len(files)} folder, {len(all_df.columns)} kolom (schema stable).')
print('\nDistribusi tahunan:')
print(all_df.groupby('year', dropna=False).size().to_frame('n_messages'))

Memverifikasi concat 14 file processed...
OK — total 1,603,567 baris di 14 folder, 29 kolom (schema stable).

Distribusi tahunan:
      n_messages
year            
2016      120454
2017      101812
2018      114866
2019      187072
2020      163577
2021      152358
2022      157225
2023      170543
2024      182574
2025      197781
2026       55305


In [6]:
# Sel 5: Hash gabungan seluruh data/processed/*.parquet → catat ke configs/experiment.yaml
import hashlib
from src.pipeline.writer import sha256_of_file

h = hashlib.sha256()
for f in files:
    h.update(sha256_of_file(f).encode('ascii'))
    h.update(b'\n')
data_processed_hash = h.hexdigest()
print(f'data_processed_hash = {data_processed_hash}')

# Update yaml manifest (in-place, hanya field data_processed_hash)
import yaml
cfg_path = Path('configs/experiment.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['data']['data_processed_hash'] = data_processed_hash
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(f'Tersimpan ke {cfg_path}')
run_log.add_output(cfg_path)

data_processed_hash = f692bb679a878a669de98a752376abf14137aff2ec2412bb0b7933591c9b3f3b
Tersimpan ke configs\experiment.yaml


In [7]:
# Sel 6: Run log
run_log.save('reports/run_log.csv')

[run_log] 01_data_pipeline → 1603.34s, 29 outputs, 0 warnings → reports\run_log.csv
